# Module 5 — Building a Custom DataLoader for Large Datasets

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: PyTorch, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 4 (PyTorch Tensors vs. NumPy)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Large Dataset Generation](#3-synthetic-large-dataset-generation)
4. [Custom Dataset Class](#4-custom-dataset-class)
5. [DataLoader Basics](#5-dataloader-basics)
6. [Batching Strategies](#6-batching-strategies)
7. [Multi-Worker Loading](#7-multi-worker-loading)
8. [Memory-Mapped Files](#8-memory-mapped-files)
9. [Performance Benchmarking](#9-performance-benchmarking)
10. [Visualization Dashboard](#10-visualization-dashboard)
11. [Key Business Takeaway](#11-key-business-takeaway)

---
## §1 · Business Context

### Why do custom DataLoaders matter at scale?

At **Revolut** and **Yandex**, datasets often exceed available RAM:

| Company | Dataset | Size | Challenge |
|---------|---------|------|-----------|
| **Revolut** | 1 year of transactions | 50 GB | Can't load all at once |
| **Yandex** | Search query logs | 200 GB | Must stream from disk |
| **Both** | User embeddings | 10 GB | Need batch-wise GPU transfer |

**The solution**: PyTorch's `Dataset` + `DataLoader` pattern:
- **Dataset**: Defines how to load a single sample (lazy loading)
- **DataLoader**: Handles batching, shuffling, and multi-worker parallelism

In this notebook we will:
1. Generate a 5M-row synthetic dataset (simulating a large CSV).
2. Build a custom `Dataset` class that loads rows on-demand.
3. Explore batching, shuffling, and multi-worker loading.
4. Benchmark different loading strategies.
5. Learn memory-mapped file techniques for out-of-core data.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import time
import os
import warnings

np.random.seed(42)
torch.manual_seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  PyTorch {torch.__version__}  |  Pandas {pd.__version__}")

---
## §3 · Synthetic Large Dataset Generation

We simulate **5 million rows** of user-feature data (like a preprocessed feature store).

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_ROWS = 5_000_000
N_FEATURES = 20
CSV_PATH = "synthetic_large_dataset.csv"

print(f"Generating {N_ROWS:,} rows × {N_FEATURES} features …")
t0 = time.time()

# Generate synthetic features (user embeddings, transaction stats, etc.)
data = np.random.randn(N_ROWS, N_FEATURES).astype(np.float32)

# Add a target column (binary classification: churn = 1, active = 0)
target = (np.random.random(N_ROWS) < 0.15).astype(np.int64)  # 15% positive rate

# Save to CSV (simulating a large file on disk)
df_large = pd.DataFrame(data, columns=[f"feature_{i}" for i in range(N_FEATURES)])
df_large["target"] = target
df_large.to_csv(CSV_PATH, index=False)

file_size_mb = os.path.getsize(CSV_PATH) / (1024 * 1024)
print(f"✓ Generated in {time.time() - t0:.2f}s")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"  Shape: {df_large.shape}")
print(f"  Positive rate: {target.mean() * 100:.1f}%")

df_large.head()

In [ ]:
# Memory check: how much RAM would loading everything take?
estimated_memory = df_large.memory_usage(deep=True).sum() / (1024**2)
print(f"If loaded fully into memory: {estimated_memory:.1f} MB")
print(f"\n💡 For 50 GB datasets, this approach would crash — we need lazy loading!")

---
## §4 · Custom Dataset Class

### 4.1 — Basic Dataset (In-Memory)

For datasets that fit in RAM, we load once in `__init__` and index in `__getitem__`.

In [ ]:
class SimpleDataset(Dataset):
    """In-memory dataset: loads entire CSV into RAM."""
    
    def __init__(self, csv_path):
        print(f"Loading {csv_path} into memory …")
        t0 = time.time()
        df = pd.read_csv(csv_path)
        self.features = torch.tensor(df.drop(columns=["target"]).values, dtype=torch.float32)
        self.targets = torch.tensor(df["target"].values, dtype=torch.long)
        print(f"  Loaded {len(self)} samples in {time.time() - t0:.2f}s")
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

# Test it
dataset_simple = SimpleDataset(CSV_PATH)
X_sample, y_sample = dataset_simple[0]
print(f"\nSample 0:")
print(f"  Features shape: {X_sample.shape}")
print(f"  Target: {y_sample.item()}")

### 4.2 — Lazy-Loading Dataset (Out-of-Core)

For datasets **too large for RAM**, we load only the requested row on each `__getitem__` call.

In [ ]:
class LazyDataset(Dataset):
    """Out-of-core dataset: loads rows on-demand from disk."""
    
    def __init__(self, csv_path):
        self.csv_path = csv_path
        # Read only the header and row count (fast!)
        t0 = time.time()
        with open(csv_path, "r") as f:
            self.header = f.readline().strip().split(",")
            self.num_rows = sum(1 for _ in f)
        print(f"  Initialized lazy dataset in {time.time() - t0:.2f}s (header + row count only)")
    
    def __len__(self):
        return self.num_rows
    
    def __getitem__(self, idx):
        # Read a single row from disk (slow but memory-efficient)
        row_idx = idx + 1  # +1 to skip header
        df_row = pd.read_csv(self.csv_path, skiprows=range(1, row_idx), nrows=1, header=None)
        features = torch.tensor(df_row.iloc[0, :-1].values, dtype=torch.float32)
        target = torch.tensor(df_row.iloc[0, -1], dtype=torch.long)
        return features, target

# Test it (only loads row 0, not the entire file)
print("Initializing lazy dataset …")
dataset_lazy = LazyDataset(CSV_PATH)
print(f"\nLazy dataset size: {len(dataset_lazy):,}")

X_sample, y_sample = dataset_lazy[0]
print(f"\nSample 0 (loaded on-demand):")
print(f"  Features shape: {X_sample.shape}")
print(f"  Target: {y_sample.item()}")

### 4.3 — Chunked Dataset (Balanced Approach)

Load data in chunks (e.g., 100K rows at a time) to balance memory and speed.

In [ ]:
class ChunkedDataset(Dataset):
    """Loads data in chunks to balance memory usage and speed."""
    
    def __init__(self, csv_path, chunk_size=100_000):
        self.csv_path = csv_path
        self.chunk_size = chunk_size
        self.chunks = []
        self.chunk_starts = []
        
        print(f"Loading data in chunks of {chunk_size:,} …")
        t0 = time.time()
        
        # Load in chunks
        for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
            features = torch.tensor(chunk.drop(columns=["target"]).values, dtype=torch.float32)
            targets = torch.tensor(chunk["target"].values, dtype=torch.long)
            self.chunks.append((features, targets))
            self.chunk_starts.append(i * chunk_size)
        
        self.total_samples = sum(len(c[0]) for c in self.chunks)
        print(f"  Loaded {self.total_samples:,} samples in {len(self.chunks)} chunks ({time.time() - t0:.2f}s)")
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # Find which chunk this index belongs to
        chunk_idx = 0
        for i, start in enumerate(self.chunk_starts):
            if idx < start + len(self.chunks[i][0]):
                chunk_idx = i
                break
        
        # Get the local index within the chunk
        local_idx = idx - self.chunk_starts[chunk_idx]
        features, targets = self.chunks[chunk_idx]
        return features[local_idx], targets[local_idx]

# Test it
dataset_chunked = ChunkedDataset(CSV_PATH, chunk_size=1_000_000)
X_sample, y_sample = dataset_chunked[0]
print(f"\nSample 0:")
print(f"  Features shape: {X_sample.shape}")
print(f"  Target: {y_sample.item()}")

---
## §5 · DataLoader Basics

The `DataLoader` wraps a `Dataset` and provides:
- **Batching**: Group samples into batches for efficient GPU processing
- **Shuffling**: Randomize order each epoch (important for training)
- **Multi-worker**: Parallel data loading across CPU cores

In [ ]:
# Create a DataLoader from the simple dataset
dataloader = DataLoader(
    dataset_simple,
    batch_size=256,           # 256 samples per batch
    shuffle=True,             # shuffle at each epoch
    num_workers=0,            # 0 = main process only (use 0 for Jupyter on some systems)
    drop_last=False,          # keep the last incomplete batch
)

print(f"Dataset size    : {len(dataset_simple):,}")
print(f"Batch size      : 256")
print(f"Number of batches: {len(dataloader):,}")

# Iterate over one epoch
t0 = time.time()
for batch_idx, (X_batch, y_batch) in enumerate(dataloader):
    if batch_idx == 0:
        print(f"\nFirst batch:")
        print(f"  X shape: {X_batch.shape}  (batch_size × n_features)")
        print(f"  y shape: {y_batch.shape}  (batch_size,)")
        print(f"  y distribution: {y_batch.float().mean().item():.3f} (fraction positive)")
    if batch_idx >= 2:
        break

print(f"\n✓ Loaded 3 batches in {time.time() - t0:.4f}s")

---
## §6 · Batching Strategies

### 6.1 — Batch Size vs. GPU Memory

In [ ]:
# Simulate different batch sizes and their memory footprint
batch_sizes = [32, 64, 128, 256, 512, 1024, 2048]
n_features = 20
bytes_per_float = 4

results = []
for bs in batch_sizes:
    # Memory for one batch (features + targets)
    mem_mb = (bs * n_features * bytes_per_float + bs * 8) / (1024**2)  # 8 bytes for int64
    num_batches = (N_ROWS + bs - 1) // bs
    results.append({
        "Batch Size": bs,
        "Memory per Batch (MB)": mem_mb,
        "Number of Batches": num_batches,
    })

df_batch = pd.DataFrame(results)
print(df_batch.to_string(index=False))

print("\n💡 Rule of thumb:")
print("   - Start with batch_size = 256 for most tasks")
print("   - Increase to 512–1024 if GPU memory allows")
print("   - Decrease to 32–64 for very large models or limited GPU")

### 6.2 — Impact of Batch Size on Training Speed

In [ ]:
# Simulate training loop with different batch sizes
batch_sizes_test = [64, 128, 256, 512, 1024]
times = []

for bs in batch_sizes_test:
    dl = DataLoader(dataset_simple, batch_size=bs, shuffle=False, num_workers=0)
    
    t0 = time.time()
    for X_batch, y_batch in dl:
        # Simulate a forward pass (just move to "GPU" — here we stay on CPU)
        _ = X_batch * 2 + 1  # dummy computation
    elapsed = time.time() - t0
    times.append(elapsed)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=batch_sizes_test, y=times,
    mode="lines+markers",
    name="Epoch Time",
    line=dict(color="#636EFA", width=2.5),
))

fig.update_layout(
    title="Epoch Time vs. Batch Size (5M samples)",
    xaxis_title="Batch Size",
    yaxis_title="Time per Epoch (seconds)",
    height=420,
)
fig.show()

print("💡 Larger batches = fewer iterations = faster epoch (up to a point)")
print("   But very large batches may hurt generalization (see Keskar et al., 2017)")

---
## §7 · Multi-Worker Loading

`num_workers > 0` enables parallel data loading across CPU cores.

⚠️ **Caveat**: Multi-processing in Jupyter can be tricky. Use `num_workers=0` if you encounter issues.

In [ ]:
# Benchmark different num_workers settings
worker_counts = [0, 2, 4]
worker_times = []

for nw in worker_counts:
    dl = DataLoader(
        dataset_simple,
        batch_size=256,
        shuffle=True,
        num_workers=nw,
        persistent_workers=(nw > 0),  # keep workers alive across epochs
    )
    
    t0 = time.time()
    for X_batch, y_batch in dl:
        pass  # just iterate
    elapsed = time.time() - t0
    worker_times.append(elapsed)
    print(f"num_workers={nw}: {elapsed:.2f}s")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"{nw} workers" for nw in worker_counts],
    y=worker_times,
    marker_color=["#EF553B", "#FFA15A", "#00CC96"],
))

fig.update_layout(
    title="DataLoader Speed vs. Number of Workers",
    yaxis_title="Time per Epoch (seconds)",
    height=380,
)
fig.show()

print("\n💡 Multi-worker loading can speed up data loading by 2–4×")
print("   But it has overhead — only use if data loading is the bottleneck")

---
## §8 · Memory-Mapped Files

For **very large datasets** (50+ GB), memory-mapped files (`np.memmap`) let you
access disk data as if it were in RAM — the OS loads only the requested pages.

In [ ]:
# Create a memory-mapped file
MEMMAP_PATH = "large_data.dat"
SHAPE = (N_ROWS, N_FEATURES)
DTYPE = np.float32

# Generate and save as binary
print(f"Creating memory-mapped file ({SHAPE[0]:,} × {SHAPE[1]}) …")
t0 = time.time()

# Write data to binary file
data_memmap = np.random.randn(*SHAPE).astype(DTYPE)
data_memmap.tofile(MEMMAP_PATH)

file_size_gb = os.path.getsize(MEMMAP_PATH) / (1024**3)
print(f"✓ Created {MEMMAP_PATH} ({file_size_gb:.2f} GB) in {time.time() - t0:.2f}s")

# Load as memory-mapped (doesn't load into RAM yet!)
mmapped = np.memmap(MEMMAP_PATH, dtype=DTYPE, mode="r", shape=SHAPE)
print(f"\nMemory-mapped array:")
print(f"  Shape: {mmapped.shape}")
print(f"  Dtype: {mmapped.dtype}")
print(f"  RAM used: ~0 MB (data is still on disk)")

# Access a slice (only loads those rows into RAM)
t0 = time.time()
slice_data = mmapped[1000:2000]  # loads only 1000 rows
print(f"\nAccessing rows 1000–2000: {time.time() - t0:.4f}s")
print(f"  Slice shape: {slice_data.shape}")
print(f"  RAM used: {slice_data.nbytes / 1024:.1f} KB")

### 8.2 — MemmapDataset for PyTorch

In [ ]:
class MemmapDataset(Dataset):
    """Dataset backed by memory-mapped file — ideal for >RAM datasets."""
    
    def __init__(self, memmap_path, shape, dtype=np.float32, targets=None):
        self.data = np.memmap(memmap_path, dtype=dtype, mode="r", shape=shape)
        self.targets = targets  # optional: separate target array
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        features = torch.from_numpy(self.data[idx].copy())  # copy to avoid memmap issues
        target = torch.tensor(self.targets[idx], dtype=torch.long) if self.targets is not None else -1
        return features, target

# Create targets array (in-memory, small)
targets = (np.random.random(N_ROWS) < 0.15).astype(np.int64)

# Initialize memmap dataset
dataset_memmap = MemmapDataset(MEMMAP_PATH, SHAPE, targets=targets)
print(f"MemmapDataset initialized: {len(dataset_memmap):,} samples")

# Test loading
X_sample, y_sample = dataset_memmap[0]
print(f"\nSample 0:")
print(f"  Features shape: {X_sample.shape}")
print(f"  Target: {y_sample.item()}")

# Benchmark iteration
dl_memmap = DataLoader(dataset_memmap, batch_size=256, shuffle=True, num_workers=0)
t0 = time.time()
for i, (X_batch, y_batch) in enumerate(dl_memmap):
    if i >= 100:
        break
elapsed = time.time() - t0
print(f"\n✓ Loaded 100 batches in {elapsed:.2f}s ({elapsed / 100 * 1000:.1f} ms/batch)")

---
## §9 · Performance Benchmarking

Comparing all dataset strategies on iteration speed.

In [ ]:
# Benchmark different strategies
strategies = {
    "Simple (in-memory)": dataset_simple,
    "Chunked (1M chunks)": dataset_chunked,
    "Memmap (disk-backed)": dataset_memmap,
}

benchmark_results = []

for name, dataset in strategies.items():
    dl = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)
    
    t0 = time.time()
    for i, (X_batch, y_batch) in enumerate(dl):
        if i >= 100:
            break
    elapsed = time.time() - t0
    
    benchmark_results.append({
        "Strategy": name,
        "Time (100 batches, s)": elapsed,
        "ms per batch": elapsed / 100 * 1000,
    })

df_bench = pd.DataFrame(benchmark_results).round(3)
print(df_bench.to_string(index=False))

print("\n💡 In-memory is fastest, but memmap enables >RAM datasets with acceptable speed")

In [ ]:
# Visualize benchmark
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_bench["Strategy"],
    y=df_bench["ms per batch"],
    marker_color=["#00CC96", "#FFA15A", "#636EFA"],
    text=df_bench["ms per batch"].apply(lambda x: f"{x:.1f}ms"),
    textposition="outside",
))

fig.update_layout(
    title="DataLoader Performance Comparison",
    yaxis_title="Time per Batch (ms)",
    height=420,
)
fig.show()

---
## §10 · Visualization Dashboard

In [ ]:
# ── 10.1 Memory Usage Over Time ──────────────────────────────────
# Simulate memory usage for different strategies
strategies_mem = ["In-Memory", "Chunked", "Lazy (on-demand)", "Memmap"]
peak_memory_mb = [estimated_memory, estimated_memory * 0.2, 50, 100]  # example values
steady_state_mb = [estimated_memory, estimated_memory * 0.2, 50, 100]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=strategies_mem,
    y=peak_memory_mb,
    marker_color=["#EF553B", "#FFA15A", "#00CC96", "#636EFA"],
    name="Peak Memory",
))

fig.update_layout(
    title="Peak Memory Usage by Loading Strategy",
    yaxis_title="Memory (MB)",
    height=400,
)
fig.show()

print("💡 Memmap and lazy loading keep memory usage low even for huge datasets")

In [ ]:
# ── 10.2 Speed vs. Memory Trade-off ──────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[estimated_memory, estimated_memory * 0.2, 50, 100],
    y=[8.5, 12.3, 45.2, 15.7],  # example ms/batch from benchmark
    mode="markers+text",
    text=["In-Memory", "Chunked", "Lazy", "Memmap"],
    textposition="top center",
    marker=dict(size=20, color=["#EF553B", "#FFA15A", "#00CC96", "#636EFA"]),
))

fig.update_layout(
    title="Speed vs. Memory Trade-off",
    xaxis_title="Peak Memory (MB)",
    yaxis_title="Time per Batch (ms)",
    height=450,
)
fig.show()

print("💡 The Pareto frontier: Memmap offers the best balance for large datasets")

---
## §11 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Dataset Size | Strategy | Use Case |
> |--------------|----------|----------|
> | **< 1 GB** | In-memory (`SimpleDataset`) | Quick prototyping, small experiments |
> | **1–10 GB** | Chunked (`ChunkedDataset`) | Feature stores, medium datasets |
> | **10–100 GB** | Memory-mapped (`MemmapDataset`) | Year-long transaction logs, embeddings |
> | **> 100 GB** | Lazy + streaming | Search logs, raw event streams |
>
> ### 💡 Revolut / Yandex Use Cases
>
> 1. **Revolut Fraud Model Training**:
>    - Dataset: 50 GB of preprocessed transaction features
>    - Solution: `MemmapDataset` + `DataLoader(num_workers=4)`
>    - Result: Train on full dataset without OOM errors
>
> 2. **Yandex Search Ranking**:
>    - Dataset: 200 GB of query-document pairs
>    - Solution: Lazy loading with on-the-fly feature extraction
>    - Result: Stream data from disk, compute features per-batch
>
> 3. **User Embedding Generation**:
>    - Dataset: 10 GB of user embeddings (50M users × 64 dims)
>    - Solution: `MemmapDataset` for inference over all users
>    - Result: Generate embeddings in batches, save to Parquet
>
> ### 📌 DataLoader Cheat Sheet
>
> ```python
> from torch.utils.data import Dataset, DataLoader
>
> class MyDataset(Dataset):
>     def __init__(self, data_path):
>         # Load or memory-map data
>         pass
>     
>     def __len__(self):
>         return num_samples
>     
>     def __getitem__(self, idx):
>         # Return (features, target) for sample idx
>         return features, target
>
> dataloader = DataLoader(
>     dataset,
>     batch_size=256,        # start with 256, tune for GPU memory
>     shuffle=True,          # True for training, False for inference
>     num_workers=4,         # 0 for debugging, 2–4 for speed
>     pin_memory=True,       # True if using GPU (faster CPU→GPU transfer)
>     drop_last=False,       # True if batch-norm and last batch is small
> )
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Always use DataLoader** — never iterate over a DataFrame in a training loop.
> 2. **Start with `num_workers=0`** for debugging, then increase to 2–4 for speed.
> 3. **Use `pin_memory=True`** when training on GPU for faster data transfer.
> 4. **Memory-mapped files** are the secret weapon for >RAM datasets.
> 5. **Batch size** affects both speed and model quality — experiment with 64, 128, 256, 512.

In [ ]:
# Cleanup temporary files
import os
for f in [CSV_PATH, MEMMAP_PATH]:
    if os.path.exists(f):
        os.remove(f)
        print(f"✓ Removed {f}")

print("\n✅ Module 5 complete!")